# 01 - Генерация fractional Brownian motion

Как пользоваться генератором авторов и что он возвращает.

Здесь только моделирование процесса: ни сноса, ни сигнатур, ни задачи
остановки. Задача остановки в одной точке - в `02-single-node.ipynb`,
перебор сетки - в `03-grid.ipynb`, проверки - в `05-checks.ipynb`.

In [ ]:
import numpy as np
from osfbm.config import Config
from osfbm import adapter
from osfbm.vendor.FBM_package import FBM

import matplotlib.pyplot as plt

C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#8a8a86"

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.8, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})

## Параметры: что и куда подаётся

Все расчёты параметризуются одним объектом `Config`. Ниже — что означает
каждое поле.

| Поле | Значение | По умолчанию |
|---|---|---|
| `T` | длина горизонта. Это **выбор единицы измерения времени**, а не ограничение: $T=1$ значит «меряем время в месяцах», тот же месяц в днях был бы $T=20$ | `1.0` |
| `n_exercise` | сколько раз за горизонт разрешено принять решение о продаже. Это $N$ из `docs/01.1`; моментов получается $N+1$, считая нулевой | `20` |
| `n_fine` | **другая** шкала: густота сетки, по которой приближаются итерированные интегралы сигнатуры. Должна делиться на `n_exercise` | `200` |
| `M_train` | число траекторий для обучения коэффициентов policy | `20000` |
| `M_test` | число **независимых** траекторий для оценки. Считать на обучающих нельзя: момент остановки перестанет быть stopping time | `20000` |
| `K` | уровень усечения сигнатуры. Признаков получается $2^{K+1}-1$, то есть рост экспоненциальный | `4` |
| `ridge` | регуляризация гребневой регрессии continuation value | `1e-9` |
| `method` | генератор fBm. Разрешён только `"cholesky"` — почему, разбирается ниже | `"cholesky"` |
| `tol_se` | Поле старого конфига для совместимости; практические границы используют отдельный `epsilon` в единицах награды | `3.0` |
| `seed` | зерно генератора | — |
| `mu_grid` | сетка значений нормированного сноса $\mu$ для скана | от $-6$ до $12$ шагом $0.5$ |
| `H_grid` | сетка значений параметра Херста | от $0.1$ до $0.9$ шагом $0.05$ |

Две шкалы времени — `n_exercise` и `n_fine` — путать нельзя. Первая говорит,
когда можно продать; вторая — насколько точно посчитаны интегралы. В notebook
авторов это `N1=10` и `N=100`.

In [ ]:
cfg_demo = Config()

print("заданные параметры")
for k, v in cfg_demo.to_dict().items():
    s = f"{v[:3]}...{v[-1]} ({len(v)} шт.)" if isinstance(v, list) else v
    print(f"  {k:12s} = {s}")

print("\nпроизводные величины")
print(f"  times          сетка t_0..t_n_fine, длина {len(cfg_demo.times)}")
print(f"  exercise_index индексы дат решения в мелкой сетке: {cfg_demo.exercise_index[:4]} ...")
print(f"  exercise_times сами даты: {cfg_demo.exercise_times[:4]} ... {cfg_demo.exercise_times[-1]}")
print(f"  sig_dim        признаков в сигнатуре при K={cfg_demo.K}: {cfg_demo.sig_dim}")
print(f"  hash           отпечаток конфига: {cfg_demo.hash}")

`Config` проверяет параметры при создании, поэтому несогласованную
конфигурацию нельзя получить молча.

In [ ]:
for kw in ({"n_fine": 201}, {"method": "daviesharte"}):
    try:
        Config(**kw)
    except ValueError as e:
        print(f"Config({kw}) ->\n    {e}\n")

## Аргументы функций генерации

| Вызов | Аргументы | Возвращает |
|---|---|---|
| `FBM(n, M, hurst, length, method).fbm()` | `n` - шагов, `M` - траекторий, `hurst` - $H$, `length` - горизонт $T$, `method` - алгоритм | `(fBm, dfBm, dW)`, формы `(n+1, M)` и `(n, M)` |
| `adapter.simulate_paths(H, cfg, seed)` | `H`, конфиг, зерно. Число траекторий берётся из `cfg.M_train` | `(B, dW)` уже транспонированные: `(M, n_fine+1)`, `(M, n_fine)` |
| `adapter.simulate_pair(H, cfg)` | `H` и конфиг. Зёрна выводятся из `cfg.seed` и `H` детерминированно | словарь `B_train`, `dW_train`, `B_test`, `dW_test` |

Приращения броуновского движения `dW` нужны для dual-оценки, поэтому генератор
отдаёт их вместе с траекторией. Время у него идёт по первой оси, нам удобнее
траектория на строку - отсюда транспонирование.

In [ ]:
n, M, H, T = 100, 5, 0.3, 1.0

np.random.seed(0)
B_raw, dB_raw, dW_raw = FBM(n, M, H, length=T, method="cholesky").fbm()

print("как отдаёт vendor:", B_raw.shape, dB_raw.shape, dW_raw.shape)
B, dW = B_raw.T, dW_raw.T
print("после транспонирования:", B.shape, dW.shape)
print("B[:, 0] == 0:", np.allclose(B[:, 0], 0.0))

## Почему только `method="cholesky"`

`Config` запрещает остальные методы, и это не вкусовщина.

`_cholesky` умножает шум на нижнетреугольную матрицу - преобразование
**причинное**, поэтому $B^H_{t_k}$ зависит только от первых $k$ гауссовских
чисел, и фильтрации $B^H$ и $W$ совпадают. У `_daviesharte` внутри
досоздаётся собственный шум и всё смешивается через FFT непричинно:
возвращаемое `dW` перестаёт порождать fBm, и dual становится невалидным.

Проверить причинность легко на $H=1/2$, где fBm обязан **в точности** совпасть
с накопленными приращениями $W$.

In [ ]:
np.random.seed(1)
B_half, dW_half = (a.T for a in FBM(n, M, 0.5, length=T, method="cholesky").fbm()[::2])

W = np.zeros_like(B_half)
W[:, 1:] = np.cumsum(dW_half, axis=1)
print("H=1/2: B == cumsum(dW):", np.allclose(B_half, W))
print("max |B - W| =", np.abs(B_half - W).max())

## Траектории при разных H

Параметр Херста задаёт автокорреляцию приращений - это и есть содержательный
центр работы.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
t = np.linspace(0, T, n + 1)

for ax, (h, color, label) in zip(axes, [
        (0.2, C1, "H = 0.2 - antipersistence, возврат к среднему"),
        (0.5, C2, "H = 0.5 - независимые приращения"),
        (0.8, C3, "H = 0.8 - persistence, тренды продолжаются")]):
    np.random.seed(7)
    paths = FBM(n, 4, h, length=T, method="cholesky").fbm()[0].T
    for p in paths:
        ax.plot(t, p, color=color, alpha=0.75)
    ax.axhline(0, color=MUTED, linewidth=0.8)
    ax.set_title(label, loc="left", fontsize=9)
    ax.set_xlabel("t")

axes[0].set_ylabel("$B^H_t$")
fig.suptitle("Траектории fBm: чем больше H, тем глаже путь", x=0.5, y=1.04, fontsize=11)
plt.tight_layout()

## Проверка генератора: ковариация

Эмпирическая ковариация должна сойтись к

$$\mathrm{Cov}(B^H_t, B^H_s) = \tfrac12\left(t^{2H} + s^{2H} - |t-s|^{2H}\right).$$

Это одна из блокирующих проверок; в собранном виде они все — в `05-checks.ipynb`.

In [ ]:
H_chk, M_chk, n_chk = 0.3, 20_000, 20
cfg = Config(n_fine=n_chk, n_exercise=10, M_train=M_chk)

B_chk, _ = adapter.simulate_paths(H_chk, cfg, seed=42)
tt = cfg.times

emp = (B_chk.T @ B_chk) / M_chk
theo = 0.5 * (tt[:, None] ** (2 * H_chk) + tt[None, :] ** (2 * H_chk)
              - np.abs(tt[:, None] - tt[None, :]) ** (2 * H_chk))

print(f"H = {H_chk}, M = {M_chk:,}")
print(f"max |эмпирическая - теоретическая| = {np.abs(emp - theo).max():.2e}")
print()
print("диагональ Var(B_t), должна быть t^(2H):")
for k in (5, 10, 15, 20):
    print(f"  t={tt[k]:.2f}   эмп {emp[k, k]:.4f}   теор {theo[k, k]:.4f}")

## Удобная обёртка

В остальном коде генератор вызывается через `adapter`, который берёт на себя
транспонирование, seed и выбор метода.

### Откуда берутся `train` и `test` и что именно обучается

Обе выборки - это два **независимых прогона одного и того же генератора** при
одном и том же $H$. Никаких «данных» здесь нет: траектории симулируются, а не
наблюдаются, и распределение у них одинаковое. Отличаются они только зерном
(`simulate_pair` берёт `base` и `base + 1`) и, при желании, размером
(`M_train` против `M_test`).

Обучается не процесс, а **правило остановки**. В `02-single-node.ipynb`
continuation value

$$C_{t_k} = E\bigl[\,Y_{t_{k+1}} \mid \mathcal F_{t_k}\,\bigr]$$

приближается линейной функцией сигнатуры пути,
$\hat C_{t_k} = \langle \beta_k,\ \mathrm{Sig}(X)_{t_k} \rangle$.
Коэффициенты $\beta_k$ - свой набор на каждую дату решения, всего
`n_exercise - 1` регрессий - подбираются гребневой регрессией по
**обучающим** траекториям. Это единственное, что подгоняется по выборке.

Зачем нужна отдельная тестовая выборка. Если применить policy к тем же
траекториям, по которым подобраны $\beta_k$, то решение «стоп или дальше» в
момент $t_k$ начнёт зависеть от всей выборки целиком, включая будущее самой
этой траектории. Тогда $\tau$ перестаёт быть stopping time, оценка $E[Z_\tau]$
смещается вверх и больше не является нижней оценкой $V$. На независимых
траекториях $\beta_k$ уже фиксированы, $\tau$ - честный stopping time, и
неравенство $\hat V \le V$ восстанавливается.

Про `dW`: приращения порождающего броуновского движения нужны только для dual
(верхней оценки), где по ним строится мартингал. Для primal используется один
лишь путь `B`. Возвращаются они всё равно парой, потому что генератор отдаёт
их вместе и пересчитать `dW` отдельно уже нельзя.

In [ ]:
cfg = Config(n_fine=100, n_exercise=10, M_train=1000, M_test=1000)
paths = adapter.simulate_pair(0.3, cfg)
for k, v in paths.items():
    print(f"{k:10s} {v.shape}")

print("\nсовпадают ли выборки:", np.allclose(paths["B_train"], paths["B_test"]))
print("E[B_T]  train %+.4f   test %+.4f   (теоретически 0)"
      % (paths["B_train"][:, -1].mean(), paths["B_test"][:, -1].mean()))
print("Var[B_T] train %.4f   test %.4f   (теоретически T^2H = %.4f)"
      % (paths["B_train"][:, -1].var(), paths["B_test"][:, -1].var(), 1.0))